In [ ]:
!pip3 install transformers tokenizers datasets --upgrade

In [ ]:

from datasets import load_from_disk, Dataset, load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForDocumentQuestionAnswering
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm.auto import tqdm
import string
import datasets
from glob import glob
from tqdm import tqdm
import json

print(datasets.__version__)

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple, Union

import torch
from torch import nn
from transformers import LayoutLMModel, LayoutLMPreTrainedModel
from transformers.modeling_outputs import QuestionAnsweringModelOutput as QuestionAnsweringModelOutputBase


@dataclass
class QuestionAnsweringModelOutput(QuestionAnsweringModelOutputBase):
    token_logits: Optional[torch.FloatTensor] = None


# There are three additional config parameters that this model supports, which are not part of the
# LayoutLMForQuestionAnswering in mainline transformers. These config parameters control the additional
# token classifier head.
#
# token_classification (`bool, *optional*, defaults to False):
#     Whether to include an additional token classification head in question answering
# token_classifier_reduction (`str`, *optional*, defaults to "mean")
#     Specifies the reduction to apply to the output of the cross entropy loss for the token classifier head during
#     training. Options are: 'none' | 'mean' | 'sum'. 'none': no reduction will be applied, 'mean': the weighted
#     mean of the output is taken, 'sum': the output will be summed.
# token_classifier_constant (`float`, *optional*, defaults to 1.0)
#     Coefficient for the token classifier head's contribution to the total loss. A larger value means that the model
#     will prioritize learning the token classifier head during training.
class LayoutLMForQuestionAnswering(LayoutLMPreTrainedModel):
    def __init__(self, config, has_visual_segment_embedding=True):
        super().__init__(config)
        self.num_labels = config.num_labels
        
        print("Num Labels: ", self.num_labels)
        

        self.layoutlm = LayoutLMModel(config)
        self.qa_outputs = nn.Linear(config.hidden_size, config.num_labels)

        # NOTE: We have to use getattr() here because we do not patch the LayoutLMConfig
        # class to have these extra attributes, so existing LayoutLM models may not have
        # them in their configuration.
        self.token_classifier_head = None
        if getattr(self.config, "token_classification", True):
            self.token_classifier_head = nn.Linear(config.hidden_size, 2)

        # Initialize weights and apply final processing
        self.post_init()
        self.token_classifier_reduction = "sum"
        self.token_classifier_constant = 0.1
        

    def get_input_embeddings(self):
        return self.layoutlm.embeddings.word_embeddings

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        bbox: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        start_positions: Optional[torch.LongTensor] = None,
        end_positions: Optional[torch.LongTensor] = None,
        token_labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, QuestionAnsweringModelOutput]:
        r"""
        start_positions (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for position (index) of the start of the labelled span for computing the token classification loss.
            Positions are clamped to the length of the sequence (`sequence_length`). Position outside of the sequence
            are not taken into account for computing the loss.
        end_positions (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for position (index) of the end of the labelled span for computing the token classification loss.
            Positions are clamped to the length of the sequence (`sequence_length`). Position outside of the sequence
            are not taken into account for computing the loss.

        Returns:

        Example:

        In this example below, we give the LayoutLMv2 model an image (of texts) and ask it a question. It will give us
        a prediction of what it thinks the answer is (the span of the answer within the texts parsed from the image).

        ```python
        >>> from transformers import AutoTokenizer, LayoutLMForQuestionAnswering
        >>> from datasets import load_dataset
        >>> import torch

        >>> tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlm-base-uncased", add_prefix_space=True)
        >>> model = LayoutLMForQuestionAnswering.from_pretrained("microsoft/layoutlm-base-uncased")

        >>> dataset = load_dataset("nielsr/funsd-layoutlmv3", split="train")
        >>> example = dataset[0]
        >>> question = "what's his name?"
        >>> words = example["tokens"]
        >>> boxes = example["bboxes"]

        >>> encoding = tokenizer(
        ...     question.split(), words, is_split_into_words=True, return_token_type_ids=True, return_tensors="pt"
        ... )
        >>> bbox = []
        >>> for i, s, w in zip(encoding.input_ids[0], encoding.sequence_ids(0), encoding.word_ids(0)):
        ...     if s == 1:
        ...         bbox.append(boxes[w])
        ...     elif i == tokenizer.sep_token_id:
        ...         bbox.append([1000] * 4)
        ...     else:
        ...         bbox.append([0] * 4)
        >>> encoding["bbox"] = torch.tensor([bbox])

        >>> outputs = model(**encoding)
        >>> loss = outputs.loss
        >>> start_scores = outputs.start_logits
        >>> end_scores = outputs.end_logits
        ```
        """

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.layoutlm(
            input_ids=input_ids,
            bbox=bbox,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        if input_ids is not None:
            input_shape = input_ids.size()
        else:
            input_shape = inputs_embeds.size()[:-1]

        seq_length = input_shape[1]
        # only take the text part of the output representations
        sequence_output = outputs[0][:, :seq_length]

        logits = self.qa_outputs(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()
        end_logits = end_logits.squeeze(-1).contiguous()

        total_loss = None
        if start_positions is not None and end_positions is not None:
            # If we are on multi-GPU, split add a dimension
            if len(start_positions.size()) > 1:
                start_positions = start_positions.squeeze(-1)
            if len(end_positions.size()) > 1:
                end_positions = end_positions.squeeze(-1)
            # sometimes the start/end positions are outside our model inputs, we ignore these terms
            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)

            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            total_loss = (start_loss + end_loss) / 2

        token_logits = None
        if getattr(self.config, "token_classification", True):
            token_logits = self.token_classifier_head(sequence_output)

            if token_labels is not None:
                # Loss fn expects logits to be of shape (batch_size, num_labels, 512), but model
                # outputs (batch_size, 512, num_labels), so we need to move the dimensions around
                # Ref: https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
                token_logits_reshaped = torch.movedim(token_logits, source=token_logits.ndim - 1, destination=1)
                token_loss = nn.CrossEntropyLoss(reduction=self.token_classifier_reduction)(
                    token_logits_reshaped, token_labels
                )

                total_loss += self.token_classifier_constant * token_loss

        if not return_dict:
            output = (start_logits, end_logits)
            if self.token_classification:
                output = output + (token_logits,)

            output = output + outputs[2:]

            if total_loss is not None:
                output = (total_loss,) + output

            return output

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            token_logits=token_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
def normalize_box(x1,y1,x2,y2, width, height):
    x1 = max(0, min(x1*1000//width, 1000))
    y1 = max(0, min(y1*1000//height, 1000))
    x2 = max(0, min(x2*1000//width, 1000))
    y2 = max(0, min(y2*1000//height, 1000))
    
    return x1,y1,x2,y2

def generate_dataset():
    """
    This module will combine question and answers from the data excluding image as LayoutLM is independent of Image, it
    requires only text and box embeddings
    
    """
    train_json = json.load(open("/kaggle/input/docvqa-train/train/train_v1.0.json", "r"))
    dataset_list = []
    
    for data in tqdm(train_json["data"]):
        local_dict = {}
        ans_json_name = "/kaggle/input/docvqa-train/train/ocr_results/" + data['image'].split("/")[-1].split(".")[0] + ".json"
        ans_json = json.load(open(ans_json_name,"r"))
        answer_dict = ans_json["recognitionResults"]
        question = data["question"]
        words = []
        boxes = []
        
        local_dict["id"] = "id_" + data['image'].split("/")[-1].split(".")[0]
        local_dict["question"] = question
        local_dict["answer"] = data["answers"][0]
        
        for obj in answer_dict:
            width, length = obj["width"], obj["height"]
            lines = obj['lines']

            for line in lines:
                for word in line['words']:
                    words.append(word['text'].lower())    
                    x1,y1,x2,y2,x3,y3,x4,y4 = word['boundingBox']
                    new_x1 = min([x1,x2,x3,x4])
                    new_x2 = max([x1,x2,x3,x4])
                    new_y1 = min([y1,y2,y3,y4])
                    new_y2 = max([y1,y2,y3,y4])
                    
                    box_norm = normalize_box(new_x1,new_y1,new_x2,new_y2, width, length)
                    assert new_x2>=new_x1
                    assert new_y2>=new_y1
                    assert box_norm[2]>=box_norm[0]
                    assert box_norm[3]>=box_norm[1]

                    boxes.append(box_norm)
                    
        local_dict["context"] = words
        local_dict["bbox"] = boxes
        
        dataset_list.append(local_dict)
        
    hf_data = Dataset.from_list(dataset_list)
    return hf_data

# train_data = generate_dataset()
# train_data
        
    

# DocVQA Preparation

In [ ]:
model_checkpoint = "TusharGoel/LayoutLM-Finetuned-DocVQA"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def subfinder(words_list, answer_list):
    matches = []
    start_indices = []
    end_indices = []
    for idx, i in enumerate(range(len(words_list))):
        if words_list[i] == answer_list[0] and words_list[i : i + len(answer_list)] == answer_list:
            matches.append(answer_list)
            start_indices.append(idx)
            end_indices.append(idx + len(answer_list) - 1)
    
    if matches:
        return matches[0], start_indices[0], end_indices[0]
    else:
        return None, 0, 0
    
def clean_text(text):
    replace_chars = ',.;:()/-$%&*' 
    for j in replace_chars:
        if text is not None:
            text = text.replace(j,'')
    return text

def encode_dataset(examples, max_length=512):

    # print(examples)
    questions = examples["question"]
    words = examples["context"]
    boxes = examples["bbox"]
    answers = examples["answer"]

    # encode the batch of examples and initialize the start_positions and end_positions
    encoding = dict()

    start_positions = []
    end_positions = []
    input_ids_list = []
    token_type_ids_list = []
    bboxes_list = []
    attention_mask_list = []
    token_label_list = []
    
    for idx in range(len(questions)):
        
        encoding = tokenizer(questions[idx].split(), words[idx], truncation="only_second", max_length=max_length, padding = "max_length",
                            is_split_into_words=True, return_token_type_ids=True)

        input_ids_list.append(encoding["input_ids"])
        token_type_ids_list.append(encoding["token_type_ids"])
        attention_mask_list.append(encoding["attention_mask"])
        

        # print(local_encoding.keys())
        

        # find the position of the answer in example's words
        words_example = [clean_text(word.lower()) for word in words[idx]]
        answer = answers[idx]

        cls_index = encoding["input_ids"].index(tokenizer.cls_token_id)
        match, word_idx_start, word_idx_end = subfinder(words_example, [clean_text(ans.lower()) for ans in answer.split()])
        
        
        local_label = [0 for _ in range(len(encoding.input_ids))]
        if match:
            # if match is found, use `token_type_ids` to find where words start in the encoding
            token_type_ids = encoding['token_type_ids']
            token_start_index = 0
            while token_type_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(encoding["input_ids"]) - 1
            while token_type_ids[token_end_index] != 1:
                token_end_index -= 1

            word_ids = encoding.word_ids()[token_start_index : token_end_index + 1]
            
            
            start_position = cls_index
            end_position = cls_index

            # loop over word_ids and increase `token_start_index` until it matches the answer position in words
            # once it matches, save the `token_start_index` as the `start_position` of the answer in the encoding
            
            for id in word_ids:
                if id == word_idx_start:
                    start_position = token_start_index
                else:
                    token_start_index += 1

            # similarly loop over `word_ids` starting from the end to find the `end_position` of the answer
            for id in word_ids[::-1]:
                if id == word_idx_end:
                    end_position = token_end_index
                else:
                    token_end_index -= 1
                    
            for label_idx in range(start_position, end_position+1):
                local_label[label_idx] = 1
            
            start_positions.append(start_position)
            end_positions.append(end_position)
            

        else:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        
        
        local_bbox = []

        # print("Decoding: ",tokenizer.decode(local_encoding['input_ids']))
        for i, s, w in zip(encoding.input_ids, encoding.sequence_ids(0), encoding.word_ids(0)):
            if s == 1:
                local_bbox.append(boxes[idx][w])
            elif i == tokenizer.sep_token_id:
                local_bbox.append([1000] * 4)
            else:
                local_bbox.append([0] * 4)
        
        bboxes_list.append(local_bbox)
        token_label_list.append(local_label)
    
    encoding["start_positions"] = start_positions
    encoding["end_positions"] = end_positions
    encoding["input_ids"] = input_ids_list
    encoding["token_type_ids"] = token_type_ids_list
    encoding["attention_mask"] = attention_mask_list
    encoding["bbox"] = bboxes_list
    encoding["token_labels"] = token_label_list
#     print(f"Answer Not found: {ans_not_found}/{len(questions)} = {ans_not_found}/{len(questions)}%")
    return encoding

# encoded_train_dataset = train_data.select(range(15001, 30000)).map(
#     encode_dataset, batched=True, batch_size=2, remove_columns=train_data.column_names
# )

# encoded_train_dataset
# example = train_data[0]
# encoding = tokenizer(example["question"].split(), example["context"], return_token_type_ids=True, is_split_into_words=True)
# tokenizer.decode(encoding["input_ids"])

# Squad V2 Preparation

In [ ]:
def prepare_squad_v2():
    dataset = load_dataset("squad_v2", split = "train")
    dataset = dataset.map(lambda x: {"context": x['context'].split()})
    dataset = dataset.map(lambda x: {"answer": x["answers"]["text"]})
    
    return dataset

def encode_dataset(examples, max_length=512):

    # print(examples)
    questions = examples["question"]
    words = examples["context"]
    answers = examples["answer"]

    # encode the batch of examples and initialize the start_positions and end_positions
    encoding = dict()

    start_positions = []
    end_positions = []
    input_ids_list = []
    token_type_ids_list = []
    attention_mask_list = []
    token_label_list = []
    
    for idx in range(len(questions)):
        
        encoding = tokenizer(questions[idx].split(), words[idx], truncation="only_second", max_length=max_length, padding = "max_length",
                            is_split_into_words=True, return_token_type_ids=True)

        
        words_example = [clean_text(word.lower()) for word in words[idx]]
        answer = answers[idx]
        
        if len(answer) == 0:
            continue
            
        input_ids_list.append(encoding["input_ids"])
        token_type_ids_list.append(encoding["token_type_ids"])
        attention_mask_list.append(encoding["attention_mask"])

        cls_index = encoding["input_ids"].index(tokenizer.cls_token_id)
        match, word_idx_start, word_idx_end = subfinder(words_example, [clean_text(ans.lower()) for ans in answer[0].split()])
        
        
        local_label = [0 for _ in range(len(encoding.input_ids))]
        if match:
            # if match is found, use `token_type_ids` to find where words start in the encoding
            token_type_ids = encoding['token_type_ids']
            token_start_index = 0
            while token_type_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(encoding["input_ids"]) - 1
            while token_type_ids[token_end_index] != 1:
                token_end_index -= 1

            word_ids = encoding.word_ids()[token_start_index : token_end_index + 1]
            
            
            start_position = cls_index
            end_position = cls_index

            # loop over word_ids and increase `token_start_index` until it matches the answer position in words
            # once it matches, save the `token_start_index` as the `start_position` of the answer in the encoding
            
            for id in word_ids:
                if id == word_idx_start:
                    start_position = token_start_index
                else:
                    token_start_index += 1

            # similarly loop over `word_ids` starting from the end to find the `end_position` of the answer
            for id in word_ids[::-1]:
                if id == word_idx_end:
                    end_position = token_end_index
                else:
                    token_end_index -= 1
                    
            for label_idx in range(start_position, end_position+1):
                local_label[label_idx] = 1
            
            start_positions.append(start_position)
            end_positions.append(end_position)
            

        else:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        
        
        token_label_list.append(local_label)
    
    encoding["start_positions"] = start_positions
    encoding["end_positions"] = end_positions
    encoding["input_ids"] = input_ids_list
    encoding["token_type_ids"] = token_type_ids_list
    encoding["attention_mask"] = attention_mask_list
    encoding["token_labels"] = token_label_list
#     print(f"Answer Not found: {ans_not_found}/{len(questions)} = {ans_not_found}/{len(questions)}%")
    return encoding



squad_dataset = prepare_squad_v2()
encoded_train_dataset_squad = squad_dataset.select(range(0, 10000)).map(
    encode_dataset, batched=True, batch_size=2, remove_columns=squad_dataset.column_names
)

In [ ]:
model = LayoutLMForQuestionAnswering.from_pretrained(model_checkpoint)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 30

In [ ]:
!nvidia-smi

In [ ]:
encoded_train_dataset_squad.set_format("torch")
train_dataloader = torch.utils.data.DataLoader(encoded_train_dataset_squad, batch_size=8)
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

import torch.nn as nn
if torch.cuda.device_count() > 1:
    print("Let's use", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)
model.to(device)
model.train()

In [ ]:
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    total_loss = 0
    count = 0

    for batch in train_dataloader:
        batch = {k:v.to(device) for k,v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.mean().backward()

        total_loss += loss.mean()
        count += 1

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    
    print(f'Epoch: {epoch} Loss: {total_loss/count}')
    

In [ ]:
model.save_pretrained("layoutlm-finetune")
tokenizer.save_pretrained("layoutlm-finetune")

In [ ]:
!rm -rf /kaggle/working/layoutlm-finetune

In [ ]:
!zip -r layoutlm.zip /kaggle/working/layoutlm-finetune

In [ ]:
# Evaluation
from transformers import AutoTokenizer, AutoModelForDocumentQuestionAnswering
from datasets import load_dataset

model_checkpoint = "/kaggle/working/layoutlm-finetune"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)
model_predict = LayoutLMForQuestionAnswering.from_pretrained(model_checkpoint)

model_predict.eval()
dataset = load_dataset("nielsr/funsd", split="train")
example = dataset[0]
print(example)

question = "What is the R&D Group?"
print(question)

words = example["words"]
boxes = example["bboxes"]

encoding = tokenizer(question.split(), words,
                            is_split_into_words=True, return_token_type_ids=True, return_tensors="pt")

bbox = []
for i, s, w in zip(encoding.input_ids[0], encoding.sequence_ids(0), encoding.word_ids(0)):
    if s == 1:
        bbox.append(boxes[w])
    elif i == tokenizer.sep_token_id:
        bbox.append([1000] * 4)
    else:
        bbox.append([0] * 4)
encoding["bbox"] = torch.tensor([bbox])

word_ids = encoding.word_ids(0)
outputs = model_predict(**encoding)

loss = outputs.loss
start_scores = outputs.start_logits
end_scores = outputs.end_logits

start, end = word_ids[start_scores.argmax(-1).item()], word_ids[end_scores.argmax(-1).item()]
# print(start, end)
print(" ".join(words[start : end + 1]))